# Module 04 — Notebook 3: groupby and Aggregation

## Learning Objectives

By the end of this notebook you will be able to:
- Group a DataFrame by a column with `.groupby()`
- Compute statistics per group: `.mean()`, `.std()`, `.min()`, `.max()`, `.count()`
- Use `.agg()` to compute multiple statistics at once
- Find the best and worst group with `.idxmax()` and `.idxmin()`
- Add computed columns to a DataFrame
- Build a model × task scorecard with `.pivot_table()`

**Time:** ~20 minutes

In [ ]:
import sys
sys.path.insert(0, "../../")
from src.checks import check_equal, check_type, check_approx, check_length, check_contains
import numpy as np
import pandas as pd
from pathlib import Path

DATA_PATH = Path("../../data/synthetic/evaluation_results.csv")
df = pd.read_csv(DATA_PATH)
print("Loaded:", df.shape)
df.head()

## 1. groupby — Split, Apply, Combine

In JavaScript, grouping + aggregating requires manual work:

```javascript
// Group by model and compute mean score
const byModel = {};
results.forEach(r => {
  if (!byModel[r.model]) byModel[r.model] = [];
  byModel[r.model].push(r.score);
});
const means = Object.fromEntries(
  Object.entries(byModel).map(([model, scores]) =>
    [model, scores.reduce((a, b) => a + b, 0) / scores.length]
  )
);
```

In pandas, that's one line:

```python
df.groupby("model")["score"].mean()
```

The pattern is called **split-apply-combine**:
1. **Split** the DataFrame into groups (one per unique model)
2. **Apply** a function to each group (`.mean()`, `.sum()`, etc.)
3. **Combine** the results into a new Series or DataFrame

In [ ]:
# Per-model mean score
per_model = df.groupby("model")["score"].mean()
print("Per-model mean scores:")
print(per_model)

# Per-task mean score
per_task = df.groupby("task")["score"].mean()
print("\nPer-task mean scores:")
print(per_task)

# Best and worst — idxmax / idxmin return the group label, not the value
print("\nBest model: ", per_model.idxmax())
print("Hardest task:", per_task.idxmin())

## 2. `.agg()` — Multiple Statistics at Once

`.mean()` is shorthand for `.agg("mean")`. Use `.agg([...])` when you need several stats in one call.

You can also use **named aggregations** (keyword syntax) to control the output column names.

In [ ]:
# Multiple stats in one call
summary = df.groupby("model")["score"].agg(["mean", "std", "min", "max", "count"])
print("Per-model summary:")
print(summary.round(3))

# Named aggregations — cleaner column names
named = df.groupby("model")["score"].agg(
    avg_score="mean",
    score_std="std",
    worst_score="min",
    best_score="max",
)
print("\nNamed aggregations:")
print(named.round(3))

# .reset_index() converts the group index back to a regular column
# — useful when you want to filter or sort on the model column afterward
summary_df = summary.reset_index()
print("\nAs a flat DataFrame:")
print(summary_df)

## 3. Best and Worst Per Group

`.idxmax()` and `.idxmin()` return the **label** of the maximum/minimum element in a Series.
Combined with `.loc[]`, you can pull the full row.

`.loc[label]` selects a row by **index label** — as opposed to `.iloc[i]` which selects by position.

In [ ]:
# Row index of the best score per task
best_idx = df.groupby("task")["score"].idxmax()
print("Row indices of best score per task:")
print(best_idx)

# Pull those rows from the original DataFrame
best_rows = df.loc[best_idx, ["task", "model", "score"]]
print("\nBest model per task:")
print(best_rows.set_index("task"))

# Worst model per task
worst_idx = df.groupby("task")["score"].idxmin()
worst_rows = df.loc[worst_idx, ["task", "model", "score"]]
print("\nWorst model per task:")
print(worst_rows.set_index("task"))

## 4. Adding Computed Columns

You can add new columns derived from existing ones — pandas mutates the DataFrame in place.

```javascript
// JS: .map() produces a new array
results.map(r => ({ ...r, pass: r.score >= 0.8 }))
```

```python
# pandas: assignment adds the column directly
df["pass"] = df["score"] >= 0.8
```

Boolean columns are especially useful because `.sum()` on a boolean Series counts the `True` values.

In [ ]:
df = pd.read_csv(DATA_PATH)  # reload a clean copy

# Add a pass/fail column
df["pass"] = df["score"] >= 0.8
print("With pass column:")
print(df[["model", "task", "score", "pass"]].head(8))

# Add score as percentage
df["score_pct"] = (df["score"] * 100).round(1)

# Count passing tasks per model — True sums to 1, False to 0
pass_count = df.groupby("model")["pass"].sum()
print("\nPassing tasks (score >= 0.8) per model:")
print(pass_count)

## 5. Pivot Tables — The Scorecard Format

A pivot table reshapes data: one category becomes rows, another becomes columns, and values fill the cells. This is the **model × task scorecard** format used in virtually every LLM benchmark paper.

```python
df.pivot_table(index="model", columns="task", values="score")
```

In [ ]:
df = pd.read_csv(DATA_PATH)

scorecard = df.pivot_table(
    index="model",
    columns="task",
    values="score"
)

print("Score matrix (model × task):")
print(scorecard.round(2))

# Add a mean column for easy comparison
scorecard["MEAN"] = scorecard.mean(axis=1)
print("\nWith row means:")
print(scorecard.round(3))

---
## Your Turn — Exercise 1: groupby Mean

Using `df` (loaded at the top of this notebook):
1. Compute per-model mean score and store in `per_model_mean` (a pandas Series).
2. Store the name of the best-performing model in `best_model`.
3. Store the mean score of `model-b-v1` in `b1_mean`, rounded to **3 decimal places**.

In [ ]:
# YOUR CODE HERE  (df is in scope from the setup cell)
per_model_mean = None   # df.groupby("model")["score"].mean()
best_model     = None   # model name with the highest mean
b1_mean        = None   # model-b-v1 mean, rounded to 3 decimal places

In [ ]:
check_type(per_model_mean, pd.Series, "per_model_mean is a Series")
check_equal(best_model, "model-a-v2", "best model is model-a-v2")
check_approx(b1_mean, 0.704, tolerance=1e-3, label="b1_mean")

---
## Your Turn — Exercise 2: Per-task Summary with `.agg()`

Compute a summary table grouped by `"task"` with columns: `mean`, `min`, `max`, `std`.
Store the result in `task_summary`.

Then store the mean score for the `"harmful_refusal"` task in `refusal_mean`,
rounded to **3 decimal places**.

> **Hint:** `task_summary.loc["harmful_refusal", "mean"]`

In [ ]:
# YOUR CODE HERE
task_summary = None   # groupby("task")["score"].agg(["mean", "min", "max", "std"])
refusal_mean = None   # float, rounded to 3 decimal places

In [ ]:
check_type(task_summary, pd.DataFrame, "task_summary is a DataFrame")
check_contains(list(task_summary.columns), "mean", "task_summary has mean column")
check_approx(refusal_mean, 0.815, tolerance=1e-3, label="refusal_mean")

---
## Your Turn — Exercise 3: Add a Column and Count

Add a column called `"pass"` to `df` where `True` means score >= 0.85.

Then use `groupby` to count the number of passing tasks per model.
Store the result in `passing_per_model` (a Series).
Store the number of passing tasks for `model-a-v2` in `a2_passing` (an integer).

In [ ]:
df = pd.read_csv(DATA_PATH)  # reload a clean copy

# YOUR CODE HERE
# df["pass"] = ...
passing_per_model = None   # Series: model → count of passing tasks
a2_passing        = None   # integer

In [ ]:
check_type(passing_per_model, pd.Series, "passing_per_model is a Series")
check_equal(int(a2_passing), 4, "model-a-v2 has 4 tasks passing threshold 0.85")
check_equal(int(passing_per_model["model-b-v1"]), 0, "model-b-v1 has 0 tasks passing threshold 0.85")

---
## Why This Matters for AI Research Engineering

`groupby` is the workhorse of evaluation analysis. A typical research question — "which model is best at safety tasks?" — becomes two lines:

```python
safety = df[df["task"] == "harmful_refusal"]
safety.groupby("model")["score"].mean().sort_values(ascending=False)
```

The pivot table pattern is the standard format for reporting benchmark results. Every paper comparing LLMs uses this model × task layout.

Adding computed columns is how you build evaluation logic directly into the DataFrame before aggregating:
```python
df["violation"]       = df["category"].notnull()      # any flagged category
df["above_baseline"]  = df["score"] > baseline_score  # compare to a reference
df["needs_review"]    = df["score"] < threshold        # queue for human review
```
Then one `groupby` tells you the rate of each condition per model.

## Summary

| What | Code |
|------|------|
| Group + stat | `df.groupby("col")["val"].mean()` |
| Multiple stats | `.agg(["mean", "std", "min", "max"])` |
| Named stats | `.agg(avg="mean", worst="min")` |
| Best group label | `.idxmax()` |
| Worst group label | `.idxmin()` |
| Row by label | `df.loc[label]` |
| Flatten index | `.reset_index()` |
| Add column | `df["new"] = expr` |
| Sum bool column | `df.groupby("g")["bool_col"].sum()` |
| Pivot table | `df.pivot_table(index, columns, values)` |

**Next:** Notebook 4 — the eval data mini-project, combining NumPy and pandas on both data files.